# Basic Setup Guide for Azure Machine Learning Workspace

---
## Notebook Setup

### Imports

In [2]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Workspace

import os
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from utils.consts import STUDENT_SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, DEFAULT_RESOURCE_LOCATION, azure_credential

### Consts

In [3]:
SUBSCRIPTION_ID = STUDENT_SUBSCRIPTION_ID
LOCATION = PREFERED_RESOURCE_LOCATION or DEFAULT_RESOURCE_LOCATION
RESOURCE_GROUP_NAME = "rg-dp100-labs"

### Utils

---
## Additional Setups

### How to install extensions for Azure CLI?



For example isntall ml package ` az extension add -n ml -y`. Sometimes existing python installation interferes with the AZ python and failing the installation. To check this run this:
```
cd "C:\Program Files\Microsoft SDKs\Azure\CLI2"
./python.exe -m site 
```
You should see this:
```
sys.path = [
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2',
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\python311.zip',
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\Lib',
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\Lib\\site-packages',
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\Lib\\site-packages\\win32',
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\Lib\\site-packages\\win32\\lib',
    'C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\Lib\\site-packages\\Pythonwin',
]
USER_BASE: 'C:\\Users\\hanglei\\AppData\\Roaming\\Python' (doesn't exist)
USER_SITE: 'C:\\Users\\hanglei\\AppData\\Roaming\\Python\\Python311\\site-packages' (doesn't exist)
ENABLE_USER_SITE: True
```
If you see any additional paths you can temporarly alter them (ex. rename the folder) and rerun extension isntallation.

### Register Resource Providers

Register a Provider
- example
```
az provider register --namespace Microsoft.Databricks
```

[register multiple providers](https://stackoverflow.com/questions/59246696/register-multiple-resource-providers-with-azure-cli)

---
## Resource Group

### Resource Group - Create

#### Python SDK

In [18]:
# Obtain the management object for resources.
resource_client = ResourceManagementClient(azure_credential, SUBSCRIPTION_ID)

# Provision the resource group.
rg_result = resource_client.resource_groups.create_or_update(RESOURCE_GROUP_NAME,
    { "location": LOCATION })

print(f"Provisioned resource group {rg_result.name}")

# Within the ResourceManagementClient is an object named resource_groups,
# which is of class ResourceGroupsOperations, which contains methods like
# create_or_update.
#
# The second parameter to create_or_update here is technically a ResourceGroup
# object. You can create the object directly using ResourceGroup(location=
# LOCATION) or you can express the object as inline JSON as shown here. For
# details, see Inline JSON pattern for object arguments at
# https://learn.microsoft.com/azure/developer/python/sdk
# /azure-sdk-library-usage-patterns#inline-json-pattern-for-object-arguments

print(
    f"Provisioned resource group {rg_result.name} in the {rg_result.location} region"
)

# The return value is another ResourceGroup object with all the details of the
# new group. In this case the call is synchronous: the resource group has been
# provisioned by the time the call returns.

# To update the resource group, repeat the call with different properties, such
# as tags:
rg_result = resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP_NAME,
    {
        "location": LOCATION,
        "tags": {"environment": "test", "department": "tech"},
    },
)

print(f"Updated resource group {rg_result.name} with tags")

# Optional lines to delete the resource group. begin_delete is asynchronous.
# poller = resource_client.resource_groups.begin_delete(rg_result.name)
# result = poller.result()

Provisioned resource group rg-dp100-labs
Provisioned resource group rg-dp100-labs in the polandcentral region
Updated resource group rg-dp100-labs with tags


### Azure CLI

In [19]:
# !az group create --name $RESOURCE_GROUP_NAME --location $LOCATION

---
## Create Azure Machine Learning resource

#### Python SDK

In [21]:
ml_client = MLClient(azure_credential, SUBSCRIPTION_ID, RESOURCE_GROUP_NAME)
workspace_name = "mlw-dp100-labs"
ws_basic = Workspace(
    name=workspace_name,
    location=LOCATION,
    display_name="ML Workspace for DP-100 Labs",
    description="",
)
ml_client.workspaces.begin_create(ws_basic)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
The deployment request mlw-dp100-labs-4588423 was accepted. ARM deployment URI for reference: 
https://portal.azure.com//#blade/HubsExtension/DeploymentDetailsBlade/overview/id/%2Fsubscriptions%2Fa1267753-4c98-48c1-a8e9-9c7169202ffd%2FresourceGroups%2Frg-dp100-labs%2Fproviders%2FMicrosoft.Resources%2Fdeployments%2Fmlw-dp100-labs-4588423


#### Azure CLI

In [ ]:
# !az ml workspace create --name $workspace_name -g $RESOURCE_GROUP_NAME

#### ARM template

In [7]:
# !az deployment group create --name "playgrounddeployment" --resource-group "playgroundgroup" --template-uri "https://raw.githubusercontent.com/Azure/azure-quickstart-templates/master/quickstarts/microsoft.machinelearningservices/machine-learning-workspace-vnet/azuredeploy.json" --parameters workspaceName="playgroundworkspace" location="polandcentral"

---
## AML - Data

---
## AML - Compute

### Compute Instance

[Create an Azure Machine Learning compute instance](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-create-compute-instance?view=azureml-api-2&tabs=python)

#### Create with Azure CLI

In [ ]:
# !az ml compute create --name "ci2231" --size STANDARD_DS11_V2 --type ComputeInstance -w mlw-dp100-labs -g rg-dp100-labs

#### Create with Python SDK

In [11]:
ci_basic_name

'ci-dp100-labs202506211609'

In [15]:
# Compute Instances need to have a unique name across the region.
# Here we create a unique name with current datetime
from azure.ai.ml.entities import ComputeInstance
import datetime

ci_basic_name = "ci-dp100-labs" + datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_basic = ComputeInstance(name=ci_basic_name, size="STANDARD_DS11_V2", idle_time_before_shutdown_minutes="10")
ml_client.begin_create_or_update(ci_basic).result()

ValueError: No value for given attribute

In [17]:
!az ml compute list-sizes --location $LOCATION

ERROR: the following arguments are required: --resource-group/-g, --workspace-name/-w

Examples from AI knowledge base:
https://aka.ms/cli_ref
Read more about the command in reference docs


### Compute Cluster

#### Create with Azure CLI

In [ ]:
# !az ml compute create --name "aml-cluster" --size STANDARD_DS11_V2 --max-instances 2 --type AmlCompute -w mlw-dp100-labs -g rg-dp100-labs

#### Create with Python SDK

In [ ]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
cpu_compute_target = "aml-cluster"

try:
    # let's see if the compute target already exists
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(
        f"You already have a cluster named {cpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new cpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="STANDARD_DS11_V2",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=1,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=120,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)
